# 01 - Dataset Ingestion, Data Contract, and Exploratory Data Analysis

**Project:** Naval Propulsion Clustering  
**Academic Title:** *Unsupervised Discovery of Operating and Performance Degradation Profiles in Naval Gas Turbine Propulsion Systems*  
**Phase:** Phase 1 — Ingestion, Quality Audit & Exploratory Data Analysis  

This notebook performs exploratory data analysis on the authoritative UCI Naval Propulsion Plants dataset (#316).
All heavy logic, data ingestion contracts, and plotting routines are imported from `naval_propulsion`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from naval_propulsion.data.loader import load_naval_dataset
from naval_propulsion.data.quality import analyze_data_quality
from naval_propulsion.visualization.plots import configure_plot_style

configure_plot_style()
print("naval_propulsion package and visualization aesthetics configured.")

## 1. Load Dataset via Invariant-Checked Container

We use `load_naval_dataset()`, which enforces the strict quarantine invariant: degradation targets (`kMc`, `kMt`) are isolated in `targets`, while telemetry features reside in `features`.

In [ ]:
container = load_naval_dataset()
print(f"Total samples: {len(container.features):,}")
print(f"Sensor telemetry feature count: {len(container.feature_names)}")
print(f"Quarantined target count: {len(container.target_names)}")
print("Feature names:", container.feature_names)
print("Target names:", container.target_names)
container.features.head()

## 2. Comprehensive Data Quality Profile

We execute the standardized quality diagnostic to inspect missingness, duplicate rows, constant sensors, and extreme scale disparities.

In [ ]:
full_df = pd.concat([container.features, container.targets], axis=1)
report = analyze_data_quality(full_df)

print(f"Missing values count: {sum(p.missing_count for p in report.columns.values())}")
print(f"Duplicate rows count: {report.duplicate_rows_count}")
print(f"Zero-variance constant columns: {report.constant_columns}")
print(f"Identical column pairs: {report.identical_column_pairs}")
print(f"Scale disparity ratio (max/min std): {report.scale_summary['std_ratio_max_to_min']:,.1f}x")

## 3. Descriptive Statistics Summary Table

In [ ]:
stats_df = pd.DataFrame([
    {
        "Column": p.name,
        "Min": p.min,
        "Max": p.max,
        "Mean": p.mean,
        "Median": p.median,
        "Std": p.std,
        "Skewness": p.skewness,
        "Unique": p.unique_count,
        "Is Constant": p.is_constant
    }
    for p in report.columns.values()
])
stats_df

## 4. Operating Condition Factorial Grid (`v` and `lp`)

The dataset samples ship speed in 9 distinct steps from 3 to 27 knots (step 3 knots). Each operating point has exactly 1,326 steady-state measurements spanning different decay states.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.countplot(data=full_df, x="v", color="#2b5c8f", ax=axes[0])
axes[0].set_title("Distribution of Ship Speed v [knots]")
axes[0].set_ylabel("Count")

sns.countplot(data=full_df, x=full_df["lp"].round(2), color="#3d85c6", ax=axes[1])
axes[1].set_title("Distribution of Lever Position lp")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 5. Quarantined Degradation Targets (`kMc` and `kMt`)

Compressor decay `kMc` is sampled in [0.95, 1.00] with grid 0.001 (51 levels).
Turbine decay `kMt` is sampled in [0.975, 1.00] with grid 0.001 (26 levels).
Factorial product: 9 speeds * 51 kMc * 26 kMt = 11,934 samples.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.histplot(full_df["kMc"], bins=51, kde=True, color="#c0392b", ax=axes[0])
axes[0].set_title("Compressor Decay State kMc")
axes[0].set_xlabel("kMc [0.95 to 1.00]")

sns.histplot(full_df["kMt"], bins=26, kde=True, color="#8e44ad", ax=axes[1])
axes[1].set_title("Turbine Decay State kMt")
axes[1].set_xlabel("kMt [0.975 to 1.00]")
plt.tight_layout()
plt.show()

## 6. Correlation Analysis & Strong Multicollinearity

Thermodynamic coupling results in near-perfect linear and monotonic rank correlations between operating demand (ship speed) and temperatures/pressures.

In [ ]:
telemetry_cols = [c for c in container.feature_names if c not in ("T1", "P1")]
corr_p = container.features[telemetry_cols].corr(method="pearson")

plt.figure(figsize=(11, 9))
sns.heatmap(corr_p, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True)
plt.title("Pearson Correlation Matrix (Telemetry Features)")
plt.tight_layout()
plt.show()

## 7. Operating Variance Dominance over Degradation Signatures

Notice how variation in ship speed (operating point) creates massive spread in temperatures and shaft torques, whereas variation across degradation states accounts for subtle local shifts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc1 = axes[0].scatter(full_df["v"], full_df["GTT"], c=full_df["kMc"], cmap="viridis", s=10, alpha=0.5)
plt.colorbar(sc1, ax=axes[0], label="kMc")
axes[0].set_title("Turbine Torque (GTT) vs Speed (v)")
axes[0].set_xlabel("Ship Speed v [knots]")
axes[0].set_ylabel("Shaft Torque GTT [kN m]")

sc2 = axes[1].scatter(full_df["v"], full_df["T48"], c=full_df["kMt"], cmap="plasma", s=10, alpha=0.5)
plt.colorbar(sc2, ax=axes[1], label="kMt")
axes[1].set_title("HP Turbine Exit Temp (T48) vs Speed (v)")
axes[1].set_xlabel("Ship Speed v [knots]")
axes[1].set_ylabel("T48 [C]")
plt.tight_layout()
plt.show()

## 8. Summary of Findings & Next Steps for Phase 2

1. **Zero-Variance Sensors:** `T1` and `P1` must be dropped during Phase 2 preprocessing.
2. **Collinear Redundancy:** `Ts` and `Tp` are exact clones; one should be dropped or combined.
3. **Scale Normalization Required:** Feature scales span over 7 orders of magnitude (min std ~0.01, max std >22,000).
4. **Operating Point Dominance:** Unsupervised clustering without operating-point conditioning will naturally cluster along speed/throttle settings rather than degradation states. In Phase 2/3, we must evaluate both raw standardized clustering and operating-conditioned sub-profiling.